# UdaPlay — Notebook 01: RAG Pipeline

This notebook builds and validates the **Retrieval Augmented Generation (RAG)** pipeline for UdaPlay:

1. Load game data from `data/games/games.json` (created by Notebook 00)
2. Embed each game using `sentence-transformers`
3. Index into a persistent **ChromaDB** vector database
4. Demonstrate semantic search with ranked results
5. Validate the reusable `src/` modules

> **Prerequisite:** Run `Udaplay_00_data_collection.ipynb` first, or the notebook will fall back to a live fetch of a single demo game.

## 1. Setup

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))

CHROMA_PERSIST_DIR = os.path.join(project_root, os.getenv('CHROMA_PERSIST_DIR', 'data/chroma_db'))
GAMES_PATH = os.path.join(project_root, 'data', 'games', 'games.json')

print(f'Project root:     {project_root}')
print(f'ChromaDB path:    {CHROMA_PERSIST_DIR}')
print(f'Games data path:  {GAMES_PATH}')

## 2. Load Game Data

In [ ]:
import json
import pandas as pd
from IPython.display import display
import anthropic
from tavily import TavilyClient

if os.path.exists(GAMES_PATH):
    with open(GAMES_PATH, 'r', encoding='utf-8') as f:
        games = json.load(f)
    print(f'✓ Loaded {len(games)} games from {GAMES_PATH}')
else:
    print('⚠ games.json not found — fetching a live demo game via Tavily+Claude...')
    from src.data_collector import GameDataCollector
    llm_client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
    tavily_client = TavilyClient(api_key=os.getenv('TAVILY_API_KEY'))
    collector = GameDataCollector(
        tavily_client=tavily_client,
        llm_client=llm_client,
        output_path=GAMES_PATH,
    )
    games = collector.collect_all(['Elden Ring', 'Hollow Knight', 'Stardew Valley'], delay_seconds=1.0)
    print(f'✓ Collected {len(games)} demo games')

In [ ]:
df = pd.DataFrame([
    {
        'Title': g['title'],
        'Developer': g['developer'],
        'Genre': ', '.join(g['genre']) if isinstance(g['genre'], list) else g['genre'],
        'Platforms': ', '.join(g['platforms']) if isinstance(g['platforms'], list) else g['platforms'],
        'Metacritic': g.get('metacritic_score', 0),
        'Source': g.get('source', 'internal'),
    }
    for g in games
])

print(f'Dataset: {len(df)} games')
display(df)

## 3. EmbeddingManager

Demonstrates how text is converted to dense vectors using `all-MiniLM-L6-v2`.

In [ ]:
from src.embeddings import EmbeddingManager

embedding_mgr = EmbeddingManager(backend='sentence-transformers')
print(f'Embedding backend: sentence-transformers / all-MiniLM-L6-v2')
print(f'Vector dimension:  {embedding_mgr.dimension}')

In [ ]:
import numpy as np

demo_text = 'action RPG with open world exploration'
vector = embedding_mgr.encode(demo_text)
print(f'Text:   "{demo_text}"')
print(f'Vector: {len(vector)}-dimensional')
print(f'Sample values: {[round(v, 4) for v in vector[:6]]} ...')
print(f'Norm:   {np.linalg.norm(vector):.4f}')

In [ ]:
# Cosine similarity demo
texts = [
    'open world action RPG game',
    'role playing adventure in a vast world',
    'cooking pasta in an Italian restaurant',
]
vecs = [np.array(embedding_mgr.encode(t)) for t in texts]

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print('Cosine similarity matrix:')
for i, ti in enumerate(texts):
    for j, tj in enumerate(texts):
        if j > i:
            sim = cosine_sim(vecs[i], vecs[j])
            print(f'  [{i}]↔[{j}]  {sim:.4f}  |  "{ti[:35]}..." vs "{tj[:35]}..."')

## 4. VectorStoreManager — Setup & Indexing

In [ ]:
import chromadb
from src.vector_store import VectorStoreManager

os.makedirs(CHROMA_PERSIST_DIR, exist_ok=True)
chroma_client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
vector_store = VectorStoreManager(
    chroma_client=chroma_client,
    embedding_manager=embedding_mgr,
)

stats = vector_store.get_collection_stats()
print(f'Collection: "{stats["collection_name"]}", current count: {stats["count"]}')

In [ ]:
from tqdm.notebook import tqdm

# Upsert all games — idempotent (safe to re-run)
BATCH_SIZE = 10
total_upserted = 0

for i in tqdm(range(0, len(games), BATCH_SIZE), desc='Indexing games'):
    batch = games[i:i + BATCH_SIZE]
    total_upserted += vector_store.upsert_documents(batch)

stats = vector_store.get_collection_stats()
print(f'\n✓ Indexed {total_upserted} games')
print(f'  Collection count: {stats["count"]} (unique game_ids)')

## 5. Semantic Search Demonstrations

In [ ]:
def show_results(query: str, n: int = 5):
    results = vector_store.query(query, n_results=n)
    rows = [
        {
            'Rank': i + 1,
            'Title': r.title,
            'Relevance': f'{r.relevance_score:.3f}',
            'Distance': f'{r.distance:.4f}',
            'Developer': r.metadata.get('developer', ''),
            'Genre': r.metadata.get('genre', ''),
        }
        for i, r in enumerate(results)
    ]
    print(f'\n🔍 Query: "{query}"')
    display(pd.DataFrame(rows).style.background_gradient(subset=['Relevance'], cmap='Greens'))

In [ ]:
show_results('best open world RPG game')

In [ ]:
show_results('indie game about mental health or personal journey')

In [ ]:
show_results('games developed by FromSoftware with challenging combat')

## 6. Metadata Filtering

ChromaDB supports filtering by metadata fields alongside semantic search.

In [ ]:
# Filter: only 'internal' source documents (exclude web_search entries)
results = vector_store.query(
    'action adventure game',
    n_results=5,
    where_filter={'source': {'$eq': 'internal'}},
)

print('Results with source=internal filter:')
for r in results:
    print(f'  [{r.relevance_score:.3f}] {r.title} — {r.metadata.get("developer", "")}')

## 7. Module Import Validation

Confirms that all `src/` modules are importable and return correct types.

In [ ]:
from src.embeddings import EmbeddingManager
from src.vector_store import VectorStoreManager, RetrievalResult
from src.data_collector import GameDataCollector

# Verify RetrievalResult dataclass
results = vector_store.query('RPG game', n_results=1)
assert len(results) > 0
r = results[0]
assert isinstance(r, RetrievalResult)
assert isinstance(r.game_id, str)
assert 0.0 <= r.relevance_score <= 1.0

print('✓ All src/ modules import correctly')
print('✓ RetrievalResult dataclass structure verified')
print(f'✓ Sample result: [{r.relevance_score:.3f}] {r.title}')

## Summary

| Component | Status |
|---|---|
| `EmbeddingManager` (sentence-transformers) | ✓ Operational |
| `VectorStoreManager` (ChromaDB persistent) | ✓ Operational |
| Game data indexed | ✓ See count above |
| Semantic search | ✓ Returns ranked `RetrievalResult` objects |
| Metadata filtering | ✓ Works with `where_filter` |

Proceed to **Notebook 02** to build the full agent on top of this RAG pipeline.